# Ownership graph

Draw an SVG network of model elements and their ownership relationships.

This sample uses only standard SysML v2 concepts and automatically discovers the project's `model/` or `src/` directory.

In [ ]:
from pathlib import Path
from collections import Counter, defaultdict
import syside

def find_sysml_root(start=Path.cwd()):
    """Find the nearest model/ or src/ folder containing textual SysML."""
    for directory in (start, *start.parents):
        for folder_name in ('model', 'src'):
            candidate = directory / folder_name
            if candidate.is_dir() and next(candidate.rglob('*.sysml'), None):
                return candidate
    raise FileNotFoundError('No model/ or src/ directory containing .sysml files was found')

SYSML_ROOT = find_sysml_root()
SYSML_FILES = sorted(SYSML_ROOT.rglob('*.sysml'))
model, diagnostics = syside.try_load_model([str(path) for path in SYSML_FILES])
print(f'Loaded {len(SYSML_FILES)} SysML files from {SYSML_ROOT}')

In [ ]:
from html import escape
from IPython.display import SVG, display

semantic_elements = (
    list(model.elements(syside.Usage, include_subtypes=True))
    + list(model.elements(syside.Definition, include_subtypes=True))
)

def node_name(element):
    value = element.name or element.declared_name or element.qualified_name
    return str(value) if value else '<unnamed>'

# Prefer named elements with semantic children, then fill with other named elements.
ranked = sorted(
    (item for item in semantic_elements if item.name or item.declared_name),
    key=lambda item: sum(isinstance(child, (syside.Usage, syside.Definition)) for child in item.owned_elements),
    reverse=True,
)
selected = ranked[:60]
selected_ids = {id(item) for item in selected}

def depth(element):
    value, current, seen = 0, element.owner, set()
    while current is not None and id(current) not in seen:
        seen.add(id(current)); value += 1; current = getattr(current, 'owner', None)
    return value

columns = defaultdict(list)
for item in selected:
    columns[depth(item)].append(item)
depths = sorted(columns)
positions = {}
for column_index, level in enumerate(depths):
    for row_index, item in enumerate(columns[level]):
        positions[id(item)] = (30 + column_index * 240, 30 + row_index * 58)

width = max(500, 260 * max(1, len(depths)))
height = max(240, 70 + 58 * max((len(items) for items in columns.values()), default=1))
edges = []
nodes = []
colors = {'PartUsage':'#dff4ee', 'RequirementUsage':'#e8eefb', 'ActionUsage':'#fff0dc', 'PortUsage':'#efe5fa'}
for item in selected:
    x, y = positions[id(item)]
    owner = getattr(item, 'owner', None)
    if owner is not None and id(owner) in selected_ids:
        ox, oy = positions[id(owner)]
        edges.append(f'<path d="M {ox + 190} {oy + 20} C {ox + 215} {oy + 20}, {x - 25} {y + 20}, {x} {y + 20}" fill="none" stroke="#91a7b3" stroke-width="1.5"/>')
    kind = type(item).__name__
    fill = colors.get(kind, '#f4f6f7')
    label = escape(node_name(item)[:27])
    nodes.append(f'<g><rect x="{x}" y="{y}" width="190" height="40" rx="7" fill="{fill}" stroke="#547180"/><text x="{x + 9}" y="{y + 17}" font-size="11" font-weight="700">{label}</text><text x="{x + 9}" y="{y + 32}" font-size="9" fill="#587487">{kind}</text></g>')

svg = f'''<svg xmlns="http://www.w3.org/2000/svg" width="100%" viewBox="0 0 {width} {height}" style="background:#fbfcfc;border:1px solid #d9e3e8;border-radius:10px">{''.join(edges)}{''.join(nodes)}</svg>'''
display(SVG(svg))
print(f'Displayed {len(selected)} elements and {len(edges)} ownership edges')